# Lab 07: A First Machine Learning Species Distribution Model

## BIO597 Spatial Analysis of Biodiversity

Last week we built a suitability curve ourselves. This week we will use a **random forest** to learn a relationship between amphibian records and climate.

A decision tree asks a series of questions, such as “Is annual precipitation greater than 1,000 mm?” A random forest combines many trees. We will use `RandomForestRegressor` to produce a continuous suitability score.

We will:

* prepare presence and background data with all available climate variables
* train and test a model using annual mean temperature, as we did last week
* add two precipitation variables and compare prediction error
* make a suitability map and look at variable importance


## 1. Import packages

Most of these packages are familiar. `scikit-learn` adds tools for splitting data, fitting a random forest, and checking predictions.

Run this notebook from the `docs/labs` directory, as in earlier labs. If needed, install scikit-learn with `%pip install scikit-learn` in a new cell.


In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
import rioxarray
import xarray as xr

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split


## 2. Load the prepared data

The GeoJSON contains amphibian records and their climate values. The NetCDF contains climate layers for Maine. We do not need to download or clip rasters today.


In [ ]:
amphibians = gpd.read_file("../data/amphibians+bioclim.geojson")
climate = xr.open_dataset("../data/maine_bioclim.nc")

amphibians["species"].value_counts().head(12)


## 3. Choose a species

We will use the wood frog, **Boreorana sylvatica**, again. Keep records with coordinate uncertainty below 100 m.


In [ ]:
target_species = "Boreorana sylvatica"
presence = amphibians[amphibians["species"] == target_species].copy()
presence = presence[presence["coordinateUncertaintyInMeters"] < 100].copy()

print("Number of records:", len(presence))

# Inspect the distribution of coordinate uncertainty
_ = plt.hist(presence["coordinateUncertaintyInMeters"])

## 4. Load all climate predictors

A **predictor** is a variable we use to help predict the response. Keep all 19 bioclimatic variables available during data preparation. We will choose which ones the model uses in step 9.

The names run from `bio_1` through `bio_19`. The code below makes that list; `range(1, 20)` stops before 20.


In [ ]:
climate_variables = ["bio_" + str(number) for number in range(1, 20)]

presence[climate_variables].head()

## 5. Make a table of Maine's climate cells

Scikit-learn works with tables. Here, each row is one raster cell, with all 19 climate variables available as columns. Keep coordinates for mapping and remove cells with missing climate values. We will use this same table for both models so that changing predictors does not change the available cells.


In [ ]:
grid = climate[climate_variables].to_dataframe().reset_index()
grid = grid[["x", "y"] + climate_variables]
grid = grid.dropna(subset=climate_variables)
grid.head()


## 6. Simplify the presence data to only retain bioclim variables

The bioclim variables will be the 'features' and we will create a new column
called 'observed' which will be the 'target' for learning and prediction.

Cells containing a record get **1**. Other cells get **0** and form our background pool.

In [ ]:
presence_data = presence[climate_variables]
presence_data["observed"] = 1
presence_data

## 7. Sample background cells

**A background cell is not a confirmed absence.** It is a place without a record in this dataset. We are learning to distinguish recorded environments from background environments, not estimating abundance or the true probability of occurrence.

Use the same number of background cells as presence cells. `random_state` makes the random sample repeatable. The climate values for both groups come from the same raster table.


In [ ]:
background = grid.sample(n=len(presence_data), random_state=42)
background["observed"] = 0

model_data = pd.concat([presence_data, background], ignore_index=True)

model_data["observed"].value_counts()


## 8. Define predictors and response

In scikit-learn, **`X`** usually means the predictor table and **`y`** means the response we want to predict. Here, `y` contains our 1/0 presence/background labels. Coordinates stay out of the predictor table.

Start with just **`bio_1`: annual mean temperature (°C)**, the same variable used last week. We have all climate variables available, but `features` chooses which columns the model actually uses. Later, we will expand this list to three predictors.


In [ ]:
features = ["bio_1"]
X = model_data[features]
y = model_data["observed"]

X.head()


## 9. Set aside test data

A model can fit familiar data well and still predict new data poorly. We will use 75% of the cells to **train** the model and hold back 25% to **test** it.

Split the full modeling table once, then select our predictor columns. This keeps the same training and test cells available when we add more predictors. `stratify=y` keeps similar proportions of presence and background in the two groups.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=597,
)

print("Training cells:", len(X_train))
print("Test cells:", len(X_test))


## 11. Train a random forest

`n_estimators=100` means 100 trees. `min_samples_leaf=5` prevents each tree from making its final groups too small. `.fit()` learns the relationship between climate and our labels.

Each tree predicts an average of its training labels. Averaging across trees gives a score between 0 and 1, which we will call **relative suitability**. Its meaning depends on how we chose the background data.


In [ ]:
model = RandomForestRegressor(
    n_estimators=100,
    min_samples_leaf=5,
    random_state=42,
)
model.fit(X_train, y_train)


## 12. Predict the withheld cells

`.predict()` applies the learned relationship to a new predictor table. The model has not seen these cells' labels during training.

**Mean absolute error (MAE)** is the average distance between a predicted score and its assigned 0/1 label. Lower is better. For example, a score of 0.7 at a presence cell has an error of 0.3. For comparison, predicting 0.5 everywhere gives an MAE of 0.5. This checks our assigned labels, not actual occupancy.


In [ ]:
test_prediction = model.predict(X_test)
bio1_error = mean_absolute_error(y_test, test_prediction)
print("BIO1-only MAE:", round(bio1_error, 3))

plt.hist(test_prediction[y_test == 1], alpha=0.5, label="Presence")
plt.hist(test_prediction[y_test == 0], alpha=0.5, label="Background")
plt.xlabel("Predicted suitability")
plt.ylabel("Number of test cells")
plt.title("Annual mean temperature only")
plt.legend()
plt.show()


**Discuss:** Do held-out presence cells generally receive higher scores than background cells? Why is setting aside records a better test than checking against the records used to train the model?

## 13. Add two precipitation predictors

Temperature alone may miss conditions important for amphibians. Add:

* `bio_12`: annual precipitation (mm)
* `bio_15`: precipitation seasonality (variation in rainfall through the year)

Update `features` to select these three variables from the tables we already prepared. Keep the same training cells, test cells, background sample, and forest settings. That way we are comparing the effect of changing predictors.


In [ ]:
features = ["bio_1", "bio_12", "bio_15"]

# Update the feature table and create a new test/train split
X = model_data[features]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=597,
)

three_variable_model = RandomForestRegressor(
    n_estimators=100, min_samples_leaf=5, random_state=2,
)
three_variable_model.fit(X_train, y_train)
three_variable_prediction = three_variable_model.predict(X_test)
three_variable_error = mean_absolute_error(y_test, three_variable_prediction)

print("BIO1-only MAE:", round(bio1_error, 3))
print("BIO1 + BIO12 + BIO15 MAE:", round(three_variable_error, 3))


In [ ]:
plt.hist(three_variable_prediction[y_test == 1], alpha=0.5, label="Presence")
plt.hist(three_variable_prediction[y_test == 0], alpha=0.5, label="Background")
plt.xlabel("Predicted suitability")
plt.ylabel("Number of test cells")
plt.title("Bio1, Bio12, & Bio15")
plt.legend()
plt.show()

**Interpretation:** Did adding precipitation reduce MAE? Give a biological reason why precipitation might help predict amphibian records. Adding predictors does not always improve predictions; what would a higher MAE tell us?

We are using the test cells for a planned classroom comparison. If we repeatedly tried many predictor combinations and chose the best score, we would need a fresh test set to evaluate that choice.

## 14. Make a suitability map

Use the three-variable model for our final map. After testing, fit a new forest with all modeling cells and predict across Maine. The earlier MAE describes the withheld-cell test, not a new test of this final model.


In [ ]:
X = model_data[features]
final_model = RandomForestRegressor(
    n_estimators=100, min_samples_leaf=5, random_state=597,
)
final_model.fit(X, y)
grid["suitability"] = final_model.predict(grid[features])


The following plotting recipe turns a table column back into a raster using its coordinates. `.reindex()` restores the original grid, including missing edge cells. You can reuse this recipe for other prediction columns.


In [ ]:
suitability = grid.set_index(["y", "x"])["suitability"].to_xarray()
suitability = suitability.reindex(y=climate.y, x=climate.x)

suitability.plot(vmin=0, vmax=1, cmap="viridis")
plt.title(f"Relative suitability for {target_species}")
plt.show()


## 15. Which variables did the forest use most?

The forest provides a simple importance score for each predictor. Higher values mean a variable contributed more to reducing error during tree building. The scores sum to 1.

This describes how the model used its **training data**. It does not prove that a variable causes the species' distribution. Predictors containing similar information can share importance.


In [ ]:
importance = pd.Series(final_model.feature_importances_, index=features)
importance.plot.bar()
plt.ylabel("Variable importance")
plt.title("Which climate variables did the forest use?")
plt.tight_layout()
plt.show()


## 16. Save your suitability map

Use the same raster export method as last week. The supplied climate grid uses WGS84 longitude and latitude (`EPSG:4326`).


In [ ]:
suitability = suitability.rio.write_crs("EPSG:4326")
suitability.rio.to_raster("Lab07_suitability.tif")


## Wrap-up

1. What do `.fit()` and `.predict()` do?
2. Why do we keep the same test cells when comparing predictor sets?
3. How did MAE change when we added precipitation? What might explain that result?
4. What does variable importance tell us, and what does it not establish?
5. Why is our score not a probability that the species occurs at a location?

For the assignment, repeat this comparison for another species and explain the results in biological terms.

## References

* [Scikit-learn random forest documentation](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html)
* [WorldClim bioclimatic variables](https://www.worldclim.org/data/bioclim.html)
